## Running scGPT gene expression prediction as the documentation states, using data the model has seen before to get a baseline of gene expression prediciton

### prep

In [ ]:
# download reference source files
from pathlib import Path
from urllib.request import urlopen
import sys

assert ".venv-scgpt/" in sys.executable, sys.executable
print("Python:", sys.executable)

COMMIT = "4068d67caaac1e28d56964da68e0214817e38428"

SOURCE_DIR = (
    Path.home()
    / "scgpt-cna/sanity_check"
    / f"reference_{COMMIT[:7]}"
)

base_url = f"https://raw.githubusercontent.com/bowang-lab/scGPT/{COMMIT}/"

files = [
    "examples/pretrain.py",
    "scgpt/preprocess.py",
    "scgpt/data_collator.py",
    "data/cellxgene/data_config.py",
    "data/cellxgene/README.md",
]

for relative_path in files:
    destination = SOURCE_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)

    with urlopen(base_url + relative_path, timeout=30) as response:
        content = response.read()

    if destination.exists():
        assert destination.read_bytes() == content, (
            f"Existing source differs: {destination}"
        )
    else:
        destination.write_bytes(content)

    print(f"Ready: {relative_path} ({len(content):,} bytes)")

print("\nReference revision:", COMMIT)

Python: /home/akash1/scgpt-cna/.venv-scgpt/bin/python
Ready: examples/pretrain.py (39,911 bytes)
Ready: scgpt/preprocess.py (11,103 bytes)
Ready: scgpt/data_collator.py (20,004 bytes)
Ready: data/cellxgene/data_config.py (1,310 bytes)
Ready: data/cellxgene/README.md (2,776 bytes)

Reference revision: 4068d67caaac1e28d56964da68e0214817e38428


In [2]:
import importlib.util
import json

import numpy as np
import torch
import scgpt.preprocess as installed_preprocess

# Load the historical source under a separate module name.
source_file = SOURCE_DIR / "scgpt/preprocess.py"
spec = importlib.util.spec_from_file_location(
    "historical_scgpt_preprocess", source_file
)
historical_preprocess = importlib.util.module_from_spec(spec)
spec.loader.exec_module(historical_preprocess)

print("PyTorch:", torch.__version__)
print("Historical tie-handling default:",
      historical_preprocess._digitize.__defaults__)
print("Installed tie-handling default:",
      installed_preprocess._digitize.__defaults__)

# A tiny diagnostic example: 13 count-one genes and 7 larger counts.
counts = torch.tensor(
    [1] * 13 + [2, 3, 4, 5, 6, 7, 8],
    dtype=torch.float32,
)

historical_bins = historical_preprocess.binning(counts.clone(), n_bins=51)

np.random.seed(0)
installed_bins = installed_preprocess.binning(counts.clone(), n_bins=51)

ones = counts.eq(1)
print("\nHistorical bins for count-one genes:",
      historical_bins[ones].tolist())
print("Installed bins for count-one genes:",
      installed_bins[ones].tolist())

assert historical_preprocess._digitize.__defaults__ == ("one",)
assert historical_bins[ones].eq(31).all()

# Inspect the configuration of the checkpoint we already have.
MODEL_DIR = Path.home() / "scgpt-cna/models/whole-human"
config = json.loads((MODEL_DIR / "args.json").read_text())

print("\nCheckpoint configuration:")
for key in [
    "input_style", "input_emb_style", "n_bins",
    "max_seq_len", "training_tasks", "mask_ratio",
    "USE_GENERATIVE_TRAINING", "MVC",
]:
    print(f"{key}: {config.get(key)}")

/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


PyTorch: 2.2.2+cpu
Historical tie-handling default: ('one',)
Installed tie-handling default: ('both',)

Historical bins for count-one genes: [31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31]
Installed bins for count-one genes: [14, 9, 13, 15, 18, 11, 18, 4, 2, 20, 7, 15, 14]

Checkpoint configuration:
input_style: binned
input_emb_style: continuous
n_bins: 51
max_seq_len: 1200
training_tasks: both
mask_ratio: [0.25, 0.5, 0.75]
USE_GENERATIVE_TRAINING: True
MVC: True


### Running sanity check

In [ ]:
# NOTE: run with scgpt-infernece venv


from pathlib import Path
import importlib.util
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd

ROOT = Path.home() / "scgpt-cna"
DATA = ROOT / "sanity_check/data/census_proximal_tubule_256/reference_raw.h5ad"
MODEL = ROOT / "models/whole-human"
HISTORICAL = (
    ROOT / "sanity_check/reference_4068d67/scgpt/preprocess.py"
)

print("Python:", sys.executable)
assert Path(sys.prefix).name == ".venv-scgpt", "Select the scGPT kernel."
assert HISTORICAL.exists(), f"Missing historical source: {HISTORICAL}"

adata_ref = ad.read_h5ad(DATA)
vocab = json.loads((MODEL / "vocab.json").read_text())
config = json.loads((MODEL / "args.json").read_text())

print("Loaded:", adata_ref.shape)
print("Checkpoint bins:", config["n_bins"])

# Load the historical file explicitly, avoiding installed-version binning.
spec = importlib.util.spec_from_file_location(
    "historical_scgpt_preprocess", HISTORICAL
)
historical = importlib.util.module_from_spec(spec)
spec.loader.exec_module(historical)

print("Historical digitize default:", historical._digitize.__defaults__)
assert historical._digitize.__defaults__ == ("one",)

# Match original gene symbols, retaining the first occurrence of duplicates.
symbols = adata_ref.var["feature_name"].astype(str)
in_vocab = symbols.isin(vocab)
keep = in_vocab & ~symbols.duplicated(keep="first")

print(f"Features matching vocabulary: {in_vocab.sum():,}/{adata_ref.n_vars:,}")
print(f"Unique matched symbols retained: {keep.sum():,}")

model_counts = adata_ref.layers["counts"][:, keep.to_numpy()].tocsr()
model_counts.eliminate_zeros()
assert np.all(model_counts.getnnz(axis=1) > 0)

audit = adata_ref.obs[
    ["soma_joinid", "donor_id", "total_counts", "n_genes_detected"]
].copy()

audit["model_detected_genes"] = model_counts.getnnz(axis=1)
audit["model_count_fraction"] = (
    np.asarray(model_counts.sum(axis=1)).ravel()
    / audit["total_counts"].to_numpy()
)

# Bin each cell's complete set of matched, nonzero counts.
# No truncation, masking, or predictions yet.
bin_rows = []
for i in range(adata_ref.n_obs):
    counts = model_counts.getrow(i).data
    bins = historical.binning(counts, n_bins=config["n_bins"])

    assert bins.min() >= 1
    assert bins.max() < config["n_bins"]

    ones = counts == 1
    labels, frequencies = np.unique(bins, return_counts=True)

    bin_rows.append({
        "fraction_count_one": float(ones.mean()),
        "count_one_bin": int(bins[ones][0]) if ones.any() else np.nan,
        "n_occupied_bins": len(labels),
        "most_common_bin": int(labels[frequencies.argmax()]),
        "most_common_bin_fraction": float(frequencies.max() / len(bins)),
    })

bin_audit = pd.DataFrame(bin_rows, index=audit.index)
audit = audit.join(bin_audit)

display(audit.describe().round(3))
print(
    "Cells with fewer than 1,199 matched detected genes:",
    int((audit["model_detected_genes"] < 1199).sum()),
)
display(audit.head(10))

Python: /home/akash1/scgpt-cna/.venv-scgpt/bin/python
Loaded: (256, 60664)
Checkpoint bins: 51


/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/akash1/scgpt-cna/.venv-scgpt/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


Historical digitize default: ('one',)
Features matching vocabulary: 60,664/60,664
Unique matched symbols retained: 60,664


,soma_joinid,total_counts,n_genes_detected,model_detected_genes,model_count_fraction,fraction_count_one,count_one_bin,n_occupied_bins,most_common_bin,most_common_bin_fraction
count,2.560000e+02,256.000,256.000,256.000,256.0,256.000,256.000,256.000,256.000,256.000
mean,3.078025e+07,3045.980,1011.148,1011.148,1.0,0.701,34.848,7.680,34.848,0.701
std,1.107268e+04,6444.145,810.414,810.414,0.0,0.086,4.239,2.192,4.239,0.086
min,3.076049e+07,270.000,213.000,213.000,1.0,0.377,19.000,4.000,19.000,0.377
25%,3.077106e+07,772.000,472.750,472.750,1.0,0.645,32.000,6.000,32.000,0.645
50%,3.077989e+07,1571.500,786.500,786.500,1.0,0.697,35.000,7.000,35.000,0.697
75%,3.078997e+07,3399.750,1294.250,1294.250,1.0,0.767,38.000,9.000,38.000,0.767
max,3.079982e+07,89408.000,6691.000,6691.000,1.0,0.884,44.000,18.000,44.000,0.884


Cells with fewer than 1,199 matched detected genes: 179


,soma_joinid,donor_id,total_counts,n_genes_detected,model_detected_genes,model_count_fraction,fraction_count_one,count_one_bin,n_occupied_bins,most_common_bin,most_common_bin_fraction
0,30760490,TxK4,331.0,248,248,1.0,0.850806,42,5,42,0.850806
1,30760599,Wilms3,5612.0,1631,1631,1.0,0.593501,30,10,30,0.593501
2,30760653,TTx,292.0,229,229,1.0,0.855895,42,4,42,0.855895
3,30761186,TTx,881.0,514,514,1.0,0.747082,37,6,37,0.747082
4,30761221,TTx,950.0,559,559,1.0,0.722719,36,6,36,0.722719
5,30761285,TTx,2049.0,964,964,1.0,0.661826,33,8,33,0.661826
6,30761728,VHL,6050.0,2403,2403,1.0,0.613816,31,9,31,0.613816
7,30761890,RCC2,605.0,475,475,1.0,0.812632,40,4,40,0.812632
8,30762051,Wilms1,4182.0,1447,1447,1.0,0.655149,33,9,33,0.655149
9,30762371,TxK4,1791.0,891,891,1.0,0.675645,34,8,34,0.675645


In [2]:
import sys
import torch

# Load the historical collator.
collator_path = HISTORICAL.parent / "data_collator.py"
module_name = "scgpt._sanity_historical_collator"

spec = importlib.util.spec_from_file_location(module_name, collator_path)
historical_collator = importlib.util.module_from_spec(spec)
sys.modules[module_name] = historical_collator
spec.loader.exec_module(historical_collator)

# Its relative import initially finds the installed preprocessing module.
# Explicitly replace that binding with the verified historical function.
historical_collator.binning = historical.binning
assert historical_collator.binning is historical.binning

assert config["n_bins"] == 51  # Historical collator hard-codes 51.
pad_id = vocab[config.get("pad_token", "<pad>")]
cls_id = vocab["<cls>"]
pad_value = config.get("pad_value", -2)

collator = historical_collator.DataCollator(
    do_padding=True,
    pad_token_id=pad_id,
    pad_value=pad_value,
    do_mlm=True,
    do_binning=True,
    mlm_probability=0.25,
    mask_value=config.get("mask_value", -1),
    max_length=config["max_seq_len"],
    sampling=True,
    keep_first_n_tokens=1,
    data_style="both",
)

# Reserve separate cells for baseline estimation and evaluation.
rng = np.random.default_rng(42)
permutation = rng.permutation(adata_ref.n_obs)
reference_indices = permutation[:128]
evaluation_indices = permutation[128:]

assert not set(reference_indices) & set(evaluation_indices)

# Four evaluation cells spanning sequencing depth, for input checks only.
# This deliberately selected subset is not a representative performance sample.
ordered = evaluation_indices[
    np.argsort(
        audit.iloc[evaluation_indices]["model_detected_genes"].to_numpy()
    )
]
sanity_eval_indices = ordered[[0, len(ordered)//3, 2*len(ordered)//3, -1]]

model_symbols = symbols.loc[keep].to_numpy()
token_ids = np.array([vocab[s] for s in model_symbols], dtype=np.int64)
assert len(np.unique(token_ids)) == len(token_ids)

examples = []
expected_targets = []

for i in sanity_eval_indices:
    row = model_counts.getrow(int(i))
    gene_ids = token_ids[row.indices]
    counts = row.data.astype(np.float32)

    # Independent lookup for checking every returned expression value.
    binned = historical.binning(counts.copy(), n_bins=51)
    expected_targets.append(dict(zip(gene_ids.tolist(), binned.tolist())))

    examples.append({
        "genes": torch.tensor(
            np.concatenate(([cls_id], gene_ids)), dtype=torch.long
        ),
        # Historical pretraining uses pad_value as the <cls> expression.
        "expressions": torch.tensor(
            np.concatenate(([pad_value], counts)), dtype=torch.float32
        ),
    })

torch.manual_seed(42)
np.random.seed(42)
sanity_batch = collator(examples)

checks = []
for j, i in enumerate(sanity_eval_indices):
    visible_mask = sanity_batch["pcpt_gene"][j].ne(pad_id)
    query_mask = sanity_batch["gen_gene"][j].ne(pad_id)

    visible_ids = sanity_batch["pcpt_gene"][j][visible_mask].tolist()
    visible_values = sanity_batch["pcpt_expr"][j][visible_mask].tolist()
    query_ids = sanity_batch["gen_gene"][j][query_mask].tolist()
    targets = sanity_batch["gen_expr_target"][j][query_mask].tolist()

    assert visible_ids[0] == cls_id
    assert visible_values[0] == pad_value
    assert cls_id not in query_ids
    assert set(visible_ids[1:]).isdisjoint(query_ids)
    assert len(query_ids) > 0

    lookup = expected_targets[j]
    for gene, value in zip(visible_ids[1:], visible_values[1:]):
        assert value == lookup[gene]
    for gene, value in zip(query_ids, targets):
        assert value == lookup[gene]

    checks.append({
        "cell_index": int(i),
        "detected_genes": int(model_counts.getrow(int(i)).nnz),
        "visible_genes": len(visible_ids) - 1,
        "query_genes": len(query_ids),
        "visible_padding": int((~visible_mask).sum()),
        "query_padding": int((~query_mask).sum()),
        "query_target_mean": float(np.mean(targets)),
        "query_target_std": float(np.std(targets)),
    })

print("Historical binning binding verified.")
print("Visible/query separation and all target values verified.")
print("\nBatch shapes:")
for name, tensor in sanity_batch.items():
    print(f"{name}: {tuple(tensor.shape)}")

display(pd.DataFrame(checks))

Historical binning binding verified.
Visible/query separation and all target values verified.

Batch shapes:
pcpt_gene: (4, 901)
pcpt_expr: (4, 901)
gen_gene: (4, 299)
gen_expr_target: (4, 299)


,cell_index,detected_genes,visible_genes,query_genes,visible_padding,query_padding,query_target_mean,query_target_std
0,218,213,160,53,740,246,42.660377,2.705295
1,129,588,441,147,459,152,39.319728,3.034913
2,222,1128,846,282,54,17,35.549645,5.351412
3,139,6691,900,299,0,0,28.866221,9.803530


In [ ]:
# calculate baseline values

# Use the SAME historical binning as the evaluation targets.
reference_sum = np.zeros(model_counts.shape[1], dtype=np.float64)
reference_support = np.zeros(model_counts.shape[1], dtype=np.int64)

for i in reference_indices:
    row = model_counts.getrow(int(i))
    binned = historical.binning(
        row.data.astype(np.float32).copy(), n_bins=51
    )
    reference_sum[row.indices] += binned
    reference_support[row.indices] += 1

reference_constant = float(
    reference_sum.sum() / reference_support.sum()
)

# Fall back to the reference constant for genes absent from all reference cells.
reference_gene_mean = np.full(
    model_counts.shape[1], reference_constant, dtype=np.float64
)
np.divide(
    reference_sum,
    reference_support,
    out=reference_gene_mean,
    where=reference_support > 0,
)

token_to_column = {
    int(token): column for column, token in enumerate(token_ids)
}

print("Reference cells:", len(reference_indices))
print("Evaluation cells reserved:", len(evaluation_indices))
print("Reference constant:", round(reference_constant, 3))
print(
    "Genes detected in reference cells:",
    int((reference_support > 0).sum()),
)

Reference cells: 128
Evaluation cells reserved: 128
Reference constant: 36.077
Genes detected in reference cells: 11611


In [5]:
import scgpt
import scgpt_session as _existing_session  # See path setup below if import fails

In [8]:
import importlib.util
import sys
from time import perf_counter

helper_file = (
    ROOT / "notebooks/scgpt_session/scgpt_session.py"
).resolve()
assert helper_file.is_file(), f"Helper not found: {helper_file}"

module_name = "scgpt_sanity_helper"
spec = importlib.util.spec_from_file_location(module_name, helper_file)
assert spec is not None and spec.loader is not None

sanity_session = importlib.util.module_from_spec(spec)
sys.modules[module_name] = sanity_session
spec.loader.exec_module(sanity_session)

assert Path(sanity_session.__file__).resolve() == helper_file
assert callable(sanity_session.load_model)
print("Loaded helper:", sanity_session.__file__)
torch.set_num_threads(4)

sanity_model, loaded_vocab, loaded_config = sanity_session.load_model(MODEL)
sanity_model = sanity_model.cpu().eval()

# Ensure the IDs used to construct the inputs match the loaded checkpoint.
assert loaded_config == config
assert all(loaded_vocab[s] == int(vocab[s]) for s in vocab)

metric_rows = []
sanity_predictions = {}

start = perf_counter()

with torch.inference_mode():
    for j, i in enumerate(sanity_eval_indices):
        visible_mask = sanity_batch["pcpt_gene"][j].ne(pad_id)
        query_mask = sanity_batch["gen_gene"][j].ne(pad_id)

        # Strip padding; retain <cls> at the beginning of the visible sequence.
        known_ids = sanity_batch["pcpt_gene"][j][visible_mask][None, :]
        known_values = sanity_batch["pcpt_expr"][j][visible_mask][None, :]
        query_ids = sanity_batch["gen_gene"][j][query_mask][None, :]

        # Targets stay outside the model call.
        target = (
            sanity_batch["gen_expr_target"][j][query_mask]
            .numpy().astype(np.float64)
        )

        output = sanity_model(known_ids, known_values, query_ids)

        columns = np.array([
            token_to_column[int(token)]
            for token in query_ids[0].tolist()
        ])
        visible_mean = float(known_values[0, 1:].mean())

        predictions = {
            "GEP": output["GEP"][0].cpu().numpy(),
            "MVC": output["MVC"][0].cpu().numpy(),
            "reference_constant": np.full(target.shape, reference_constant),
            "gene_mean": reference_gene_mean[columns],
            "visible_mean": np.full(target.shape, visible_mean),
        }

        sanity_predictions[int(i)] = {
            "query_token_ids": query_ids[0].numpy().copy(),
            "target": target.copy(),
            "predictions": {
                name: np.asarray(pred).copy()
                for name, pred in predictions.items()
            },
        }

        for method, pred in predictions.items():
            pred = np.asarray(pred, dtype=np.float64)
            assert pred.shape == target.shape
            assert np.isfinite(pred).all()

            metric_rows.append({
                "cell_index": int(i),
                "detected_genes": int(model_counts.getrow(int(i)).nnz),
                "method": method,
                "n_queries": len(target),
                "target_mean": target.mean(),
                "prediction_mean": pred.mean(),
                "prediction_std": pred.std(),
                "MAE": np.abs(pred - target).mean(),
                "MSE": np.square(pred - target).mean(),
                "gene_baseline_support_fraction":
                    float((reference_support[columns] > 0).mean()),
            })

        print(f"Completed {j + 1}/{len(sanity_eval_indices)} cells", flush=True)

sanity_metrics = pd.DataFrame(metric_rows)

print(f"\nElapsed: {perf_counter() - start:.1f} seconds")
display(sanity_metrics.round(3))
display(
    sanity_metrics.pivot(
        index="cell_index", columns="method", values="MSE"
    ).round(3)
)

Loaded helper: /home/akash1/scgpt-cna/notebooks/scgpt_session/scgpt_session.py
Strict load: 163 checkpoint tensors; no omitted tensors.
Completed 1/4 cells
Completed 2/4 cells
Completed 3/4 cells
Completed 4/4 cells

Elapsed: 0.4 seconds


,cell_index,detected_genes,method,n_queries,target_mean,prediction_mean,prediction_std,MAE,MSE,gene_baseline_support_fraction
0,218,213,GEP,53,42.660,43.033,2.451,1.457,3.465,1.000
1,218,213,MVC,53,42.660,42.819,2.181,1.336,3.387,1.000
2,218,213,reference_constant,53,42.660,36.077,0.000,6.583,50.654,1.000
3,218,213,gene_mean,53,42.660,39.056,5.877,4.738,36.652,1.000
4,218,213,visible_mean,53,42.660,41.744,0.000,1.983,8.159,1.000
5,129,588,GEP,147,39.320,39.585,1.972,1.653,5.921,0.959
6,129,588,MVC,147,39.320,39.460,2.014,1.618,5.777,0.959
7,129,588,reference_constant,147,39.320,36.077,0.000,3.242,19.723,0.959
8,129,588,gene_mean,147,39.320,37.217,4.925,3.980,23.639,0.959
9,129,588,visible_mean,147,39.320,40.016,0.000,2.650,9.695,0.959


method,GEP,MVC,gene_mean,reference_constant,visible_mean
cell_index,,,,,
129,5.921,5.777,23.639,19.723,9.695
139,29.166,32.753,88.824,148.111,96.948
218,3.465,3.387,36.652,50.654,8.159
222,19.016,18.821,25.704,28.916,29.004


In [9]:
# Build fresh examples: the historical collator modifies expression tensors.
def make_example(i):
    row = model_counts.getrow(int(i))
    return {
        "genes": torch.tensor(
            np.concatenate(([cls_id], token_ids[row.indices])),
            dtype=torch.long,
        ),
        "expressions": torch.tensor(
            np.concatenate(([pad_value], row.data.astype(np.float32))),
            dtype=torch.float32,
        ),
    }

torch.manual_seed(42)
np.random.seed(42)

evaluation_batch = collator([
    make_example(i) for i in evaluation_indices
])

evaluation_rows = []
start = perf_counter()

with torch.inference_mode():
    for j, i in enumerate(evaluation_indices):
        visible_mask = evaluation_batch["pcpt_gene"][j].ne(pad_id)
        query_mask = evaluation_batch["gen_gene"][j].ne(pad_id)

        known_ids = evaluation_batch["pcpt_gene"][j][visible_mask][None, :]
        known_values = evaluation_batch["pcpt_expr"][j][visible_mask][None, :]
        query_ids = evaluation_batch["gen_gene"][j][query_mask][None, :]

        target = (
            evaluation_batch["gen_expr_target"][j][query_mask]
            .numpy().astype(np.float64)
        )

        assert known_ids[0, 0].item() == cls_id
        assert known_values[0, 0].item() == pad_value
        assert set(known_ids[0, 1:].tolist()).isdisjoint(
            query_ids[0].tolist()
        )

        correct = sanity_model(known_ids, known_values, query_ids)

        # Shuffle visible expression only; preserve <cls>, gene IDs, and queries.
        generator = torch.Generator().manual_seed(10000 + int(i))
        permutation = torch.randperm(
            known_values.shape[1] - 1, generator=generator
        )
        shuffled_values = known_values.clone()
        shuffled_values[0, 1:] = known_values[0, 1:][permutation]

        shuffled = sanity_model(known_ids, shuffled_values, query_ids)

        columns = np.array([
            token_to_column[int(token)]
            for token in query_ids[0].tolist()
        ])

        predictions = {
            "GEP": correct["GEP"][0].numpy(),
            "MVC": correct["MVC"][0].numpy(),
            "GEP_shuffled": shuffled["GEP"][0].numpy(),
            "MVC_shuffled": shuffled["MVC"][0].numpy(),
            "gene_mean": reference_gene_mean[columns],
            "reference_constant": np.full(target.shape, reference_constant),
            "visible_mean": np.full(
                target.shape, float(known_values[0, 1:].mean())
            ),
        }

        for method, pred in predictions.items():
            pred = np.asarray(pred, dtype=np.float64)
            assert pred.shape == target.shape
            assert np.isfinite(pred).all()

            evaluation_rows.append({
                "cell_index": int(i),
                "soma_joinid": int(adata_ref.obs.iloc[int(i)]["soma_joinid"]),
                "donor_id": str(adata_ref.obs.iloc[int(i)]["donor_id"]),
                "detected_genes": int(model_counts.getrow(int(i)).nnz),
                "n_visible": known_ids.shape[1] - 1,
                "n_queries": len(target),
                "method": method,
                "target_mean": float(target.mean()),
                "target_std": float(target.std()),
                "prediction_mean": float(pred.mean()),
                "prediction_std": float(pred.std()),
                "MAE": float(np.abs(pred - target).mean()),
                "MSE": float(np.square(pred - target).mean()),
            })

        if (j + 1) % 16 == 0:
            print(f"Completed {j + 1}/{len(evaluation_indices)}", flush=True)

evaluation_metrics = pd.DataFrame(evaluation_rows)

# Each cell receives equal weight in this summary.
summary = evaluation_metrics.groupby("method").agg(
    n_cells=("cell_index", "nunique"),
    mean_MSE=("MSE", "mean"),
    median_MSE=("MSE", "median"),
    mean_MAE=("MAE", "mean"),
).sort_values("mean_MSE")

paired = evaluation_metrics.pivot(
    index="cell_index", columns="method", values="MSE"
)

comparisons = []
for decoder in ["GEP", "MVC"]:
    for baseline in [
        "gene_mean", "reference_constant", "visible_mean",
        f"{decoder}_shuffled",
    ]:
        improvement = paired[baseline] - paired[decoder]
        comparisons.append({
            "decoder": decoder,
            "comparison": baseline,
            "mean_MSE_improvement": improvement.mean(),
            "median_MSE_improvement": improvement.median(),
            "cells_decoder_wins": int((improvement > 0).sum()),
            "n_cells": len(improvement),
        })

display(summary.round(3))
display(pd.DataFrame(comparisons).round(3))

output_dir = ROOT / "sanity_check/results"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "evaluation_25pct_seed42.csv"
evaluation_metrics.to_csv(output_file, index=False)

print(f"Elapsed: {perf_counter() - start:.1f} seconds")
print("Saved:", output_file)

Completed 16/128
Completed 32/128
Completed 48/128
Completed 64/128
Completed 80/128
Completed 96/128
Completed 112/128
Completed 128/128


,n_cells,mean_MSE,median_MSE,mean_MAE
method,,,,
GEP,128,10.509,10.909,2.252
MVC,128,10.856,11.268,2.299
MVC_shuffled,128,15.077,13.997,3.163
GEP_shuffled,128,15.366,14.325,3.207
gene_mean,128,23.648,19.616,3.881
visible_mean,128,23.772,23.127,4.030
reference_constant,128,34.277,30.486,4.781


,decoder,comparison,mean_MSE_improvement,median_MSE_improvement,cells_decoder_wins,n_cells
0,GEP,gene_mean,13.139,7.618,128,128
1,GEP,reference_constant,23.768,18.575,128,128
2,GEP,visible_mean,13.263,11.587,126,128
3,GEP,GEP_shuffled,4.857,3.029,121,128
4,MVC,gene_mean,12.793,7.118,128,128
5,MVC,reference_constant,23.422,18.177,128,128
6,MVC,visible_mean,12.917,11.125,127,128
7,MVC,MVC_shuffled,4.221,2.377,122,128


Elapsed: 27.1 seconds
Saved: /home/akash1/scgpt-cna/sanity_check/results/evaluation_25pct_seed42.csv
